# 특징 입력 신경망 + SPRT 검증

선형 평가 특징 11개를 입력에 더한 신경망(형식 2)이 **시간 제한 대국에서도** 선형 평가보다 강한지 SPRT 로 확인합니다.

로컬에서 본 것: 깊이 4 고정 대국에서는 Elo +127 로 이겼지만, 수당 30ms 시간 제한에서는 Elo -147 로 졌습니다.
신경망 때문에 탐색이 느려지는데(깊이 7.15 -> 6.5), 정답이 30ms 탐색 점수라 그 깊이에서 신경망이 더 알려 주는 게 없었기 때문입니다.
그래서 이 노트북은 **정답을 300ms 탐색으로 만들고**(대국 시간의 10배), 수당 30ms 대국에서 SPRT 로 판정합니다.

- 런타임은 **CPU** (GPU 안 씀). 위에서부터 한 셀씩 실행하고, 끊기면 셀 1·2 를 다시 실행한 뒤 끊긴 셀부터 이어 가세요.
- 시간(코어 2개): 자가대전 200판 x 300ms 약 40분 (Drive 에 저장해 두고 다음에는 건너뜀), 학습 몇 분, 대국은 SPRT 라 결론이 나면 일찍 끝나고 최대 각 25분 안팎.
- 대국은 수 제한을 사실상 없앴습니다 (`MAX_PLIES = 1000`, 같은 국면 3번이면 무승부). 150수 제한은 무승부를 크게 늘려 차이를 가렸습니다.
- 마지막 셀이 결과와 신경망 파일을 Google Drive 의 `slider_results` 에 저장합니다.

In [ ]:
import os, json, shutil, subprocess, pathlib, time

REPO_URL = "https://github.com/Perdume/Slider.git"
BRANCH = "rust-port"
HIDDEN = 32               # 은닉 뉴런 수
GAMES = 200               # 자가대전 판 수
TEACHER_MS = 300          # 정답(탐색 점수)을 만드는 수당 시간. 대국 시간보다 훨씬 길어야 신경망이 도움이 됩니다
EPOCHS = 60
MATCH_TIME_MS = 30        # 본 비교: 수당 시간 제한 대국
MATCH_DEPTH = 4           # 참고 비교: 깊이 고정 대국 (빠름)
MATCH_MAX_GAMES = 600     # SPRT 최대 판 수 (결론이 나면 일찍 끝남)
SPRT_ELO0, SPRT_ELO1 = 0, 30   # H0: Elo 0 이하 / H1: Elo 30 이상
MAX_PLIES = 1000          # 대국 수 제한 (사실상 없음)
RUN_PLAIN = True          # 비교용으로 특징 없는 신경망도 시간 제한 대국을 할지
SEED = 1
WEIGHTS_PATH = None       # 튜닝한 weights.json 을 올렸으면 경로, 없으면 None (기본 가중치)
SAVE_TO_DRIVE = True      # 데이터와 결과를 Google Drive 에 저장 (다음 세션에서 자가대전을 건너뜀)

THREADS = os.cpu_count() or 2
WORK = pathlib.Path("/content/feat_sprt"); WORK.mkdir(parents=True, exist_ok=True)
W = WEIGHTS_PATH if WEIGHTS_PATH and os.path.exists(WEIGHTS_PATH) else "default"
os.environ["PATH"] = str(pathlib.Path.home() / ".cargo" / "bin") + os.pathsep + os.environ["PATH"]
DRIVE_DIR = pathlib.Path("/content/drive/MyDrive/slider_results")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")   # 로그인 창이 뜨면 허용
    (DRIVE_DIR / "data").mkdir(parents=True, exist_ok=True)
print("threads:", THREADS, "| 선형 가중치:", W, "| 은닉:", HIDDEN)

In [ ]:
def sh(cmd, cwd=None, env=None):
    print("$", " ".join(map(str, cmd)) if isinstance(cmd, list) else cmd, flush=True)
    subprocess.run(cmd, shell=isinstance(cmd, str), check=True, cwd=cwd, env=env)


def run(binary, args):
    cmd = [str(binary), *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        line = line.rstrip("\n"); lines.append(line); print("   ", line, flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"실행 실패 (exit {p.returncode}): {cmd[0]}")
    return lines

## 3. Rust 설치와 코드 받기

In [ ]:
if shutil.which("cargo") is None:
    sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
REPO = WORK / "Slider"
if REPO.exists():
    sh(["git", "-C", REPO, "pull", "--ff-only"])
else:
    sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO])
sh(["git", "-C", REPO, "log", "--oneline", "-1"])

## 4. 빌드

In [ ]:
TARGET = WORK / f"target_{HIDDEN}"
sh(["cargo", "build", "--release", "--bin", "selfplay", "--bin", "trainnet", "--bin", "matchup"],
   cwd=REPO / "rust", env={**os.environ, "SLIDER_HIDDEN": str(HIDDEN), "CARGO_TARGET_DIR": str(TARGET)})
BIN = TARGET / "release"
print("빌드 완료:", BIN)

## 5. 자가대전 데이터 (정답 = TEACHER_MS 탐색 점수)
이미 만든 파일이 이 세션, Drive, 이전 노트북 폴더 중에 있으면 그걸 씁니다.

In [ ]:
name = f"selfplay_{GAMES}g_{TEACHER_MS}ms.jsonl"
data = WORK / name
candidates = [DRIVE_DIR / "data" / name, pathlib.Path("/content/hidden_cmp") / name]
if not (data.exists() and data.stat().st_size > 0):
    for c in candidates:
        if c.exists() and c.stat().st_size > 0:
            shutil.copy(c, data)
            print("기존 데이터 사용:", c)
            break
if data.exists() and data.stat().st_size > 0:
    print("데이터:", data, f"({data.stat().st_size / 1e6:.1f}MB)")
else:
    t0 = time.time()
    run(BIN / "selfplay", ["--games", GAMES, "--time-ms", TEACHER_MS, "--random-plies", 4, "--epsilon", 0.05,
                           "--threads", THREADS, "--seed", SEED, "--weights", W, "--positions", "yes", "--out", data])
    print(f"자가대전 {time.time() - t0:.0f}초")
    if SAVE_TO_DRIVE:
        shutil.copy(data, DRIVE_DIR / "data" / name)
        print("Drive 에 저장:", DRIVE_DIR / "data" / name)

## 6. 학습: 특징 입력 신경망과 (비교용) 위치 입력만 쓰는 신경망

In [ ]:
nets, train_logs = {}, {}


def train(label, features):
    net = WORK / f"net_{label}_{HIDDEN}.txt"
    lines = run(BIN / "trainnet", ["--data", data, "--weights", W, "--out", net, "--epochs", EPOCHS, "--lambda", 1.0,
                                   "--features", "yes" if features else "no", "--seed", SEED])
    best = [l for l in lines if "가장 좋은 검증 손실" in l]
    nets[label], train_logs[label] = net, (best[-1].strip() if best else "")


train("feat", True)
if RUN_PLAIN:
    train("plain", False)
for k, v in train_logs.items():
    print(f"{k}: {v}")

## 7. SPRT 대국 함수 (신경망 vs 선형 평가)

In [ ]:
results = {}


def match(label, mode):
    limit = ["--time-ms", MATCH_TIME_MS] if mode == "time" else ["--time-ms", 100000, "--depth", MATCH_DEPTH]
    lines = run(BIN / "matchup", ["--a", W, "--net-a", nets[label], "--b", W, "--games", MATCH_MAX_GAMES, *limit,
                                  "--max-plies", MAX_PLIES, "--sprt-elo0", SPRT_ELO0, "--sprt-elo1", SPRT_ELO1,
                                  "--threads", THREADS, "--seed", SEED + 1000])
    r = json.loads([l for l in lines if l.startswith("RESULT ")][-1][len("RESULT "):])
    results[(label, mode)] = r
    print(f"-> {label} / {mode}: {r['aWins']}승 {r['draws']}무 {r['bWins']}패 ({r['games']}판), Elo {r['elo']:+.0f} "
          f"({r['eloLow']:+.0f} ~ {r['eloHigh']:+.0f}), SPRT {r['sprt']}")

## 8. 참고: 특징 신경망, 깊이 고정 (몇 분)

In [ ]:
match("feat", "depth")

## 9. 본 비교: 특징 신경망, 수당 시간 제한

In [ ]:
match("feat", "time")

## 10. 비교용: 위치 입력만 쓰는 신경망, 수당 시간 제한

In [ ]:
if RUN_PLAIN:
    match("plain", "time")
else:
    print("RUN_PLAIN = False 라서 건너뜀")

## 11. 요약 + Google Drive 저장
`H1` = 신경망이 Elo 30 이상 강함, `H0` = 그만큼 강하지 않음, `undecided` = 최대 판 수까지 결론 없음.

In [ ]:
settings = {"HIDDEN": HIDDEN, "GAMES": GAMES, "TEACHER_MS": TEACHER_MS, "EPOCHS": EPOCHS, "MATCH_TIME_MS": MATCH_TIME_MS,
            "MATCH_DEPTH": MATCH_DEPTH, "MATCH_MAX_GAMES": MATCH_MAX_GAMES, "SPRT": [SPRT_ELO0, SPRT_ELO1],
            "MAX_PLIES": MAX_PLIES, "SEED": SEED, "weights": W, "threads": THREADS}
lines = [f"설정: {settings}", f"데이터: {data.name} ({data.stat().st_size / 1e6:.1f}MB)"]
for k, v in train_logs.items():
    lines.append(f"학습 {k}: {v}")
for (label, mode), r in results.items():
    cond = f"수당 {MATCH_TIME_MS}ms" if mode == "time" else f"깊이 {MATCH_DEPTH} 고정"
    lines.append(f"대국 {label} / {cond}: {r['aWins']}승 {r['draws']}무 {r['bWins']}패 ({r['games']}판), 점수 {r['score']:.3f}, "
                 f"Elo {r['elo']:+.0f} ({r['eloLow']:+.0f} ~ {r['eloHigh']:+.0f}), SPRT {r['sprt']} (LLR {r['llr']:.2f})")
text = "\n".join(lines + ["", "JSON:", json.dumps({"settings": settings, "train": train_logs,
                                                    "results": {f"{l}/{m}": r for (l, m), r in results.items()}},
                                                   ensure_ascii=False, indent=1)])
print("\n".join(lines))
if SAVE_TO_DRIVE:
    out = DRIVE_DIR / f"slider_feature_sprt_{GAMES}g_{TEACHER_MS}ms_vs{MATCH_TIME_MS}ms.txt"
    out.write_text(text, encoding="utf-8")
    for label, net in nets.items():
        shutil.copy(net, DRIVE_DIR / f"{net.stem}_{GAMES}g_{TEACHER_MS}ms.txt")
    print("\nDrive 에 저장:", out)